In [ ]:
from transformers import T5EncoderModel, T5Tokenizer
from Bio.PDB import PDBParser
from Bio import pairwise2
from tqdm import tqdm
import pandas as pd
import numpy as np
import warnings
import requests
import shutil
import pickle
import torch
import time
import os
import re
import gc

pdb_parser = PDBParser(QUIET=True)

DSSP = "./dssp-2.0.4/mkdssp"
SEQ_EMB_FODLER = "./seq_emb"
STRUCTURE_FOLDER = "./structure"
ALL_SEQS = "./all_receptor.fasta"
PROTT5_PATH = "Rostlab/prot_t5_xl_uniref50"

BIOLIP_RECEPTOR_FOLDER = "./receptor" # PDBs
BIOLIP_PEPTIDE_FOLDER = "./peptide" #PDBs

gc.enable()
warnings.filterwarnings("ignore", module="Bio")

In [ ]:
BIOLIP_FILE = "./BioLiP.txt"
BIOLIP_HEADER = [
    "pdb_id",
    "receptor_chain",
    "resolution",
    "binding_site",
    "ligand_ccd_id",
    "ligand_chain",
    "ligand_serial_num",
    "binding_site_pdb", # pocket
    "binding_site_reorder",
    "catalyst_site_pdb",
    "catalyst_site_reorder",
    "enzyme_class_id",
    "go_term_id",
    "binding_affinity_literature",
    "binding_affinity_binding_moad",
    "binding_affinity_pdbind_cn",
    "binding_affinity_binding_db",
    "uniprot_db",
    "pubmed_id",
    "ligand_res_num",
    "receptor_seq"
]
ALL_LIGAND_SEQ_FILE = "./all_ligand_seq.pkl"

with open(ALL_LIGAND_SEQ_FILE, "rb") as f:
    all_ligand_seq = pickle.load(f)

complexes = pd.read_csv(BIOLIP_FILE, sep="\t", names=BIOLIP_HEADER)
complexes.drop_duplicates(subset="pdb_id", inplace=True)
complexes = complexes.loc[complexes.resolution<5]
complexes.reset_index(drop=True, inplace=True)
complexes["ligand_seq"] = [l[1] for l in all_ligand_seq]

/tmp/ipykernel_3416164/3950718512.py:30: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  complexes = pd.read_csv(BIOLIP_FILE, sep="\t", names=BIOLIP_HEADER)


In [ ]:
data_split = torch.load("./data_split.pt")
train_ids = data_split["train"]
test_ids = data_split["test"]
val_ids = data_split["val"]

In [3]:
d3to1 = {
    'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
    'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 
    'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 
    'ALA': 'A', 'VAL':'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'
}

non_standard_residue_substitutions = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA', 'ALO':'THR', 'ALY':'LYS', 'ARM':'ARG',
    'ASA':'ASP', 'ASB':'ASP', 'ASK':'ASP', 'ASL':'ASP', 'ASQ':'ASP', 'AYA':'ALA', 'BCS':'CYS', 'BHD':'ASP', 'BMT':'THR', 'BNN':'ALA',
    'BUC':'CYS', 'BUG':'LEU', 'C5C':'CYS', 'C6C':'CYS', 'CAS':'CYS', 'CCS':'CYS', 'CEA':'CYS', 'CGU':'GLU', 'CHG':'ALA', 'CLE':'LEU', 'CME':'CYS',
    'CSD':'ALA', 'CSO':'CYS', 'CSP':'CYS', 'CSS':'CYS', 'CSW':'CYS', 'CSX':'CYS', 'CXM':'MET', 'CY1':'CYS', 'CY3':'CYS', 'CYG':'CYS',
    'CYM':'CYS', 'CYQ':'CYS', 'DAH':'PHE', 'DAL':'ALA', 'DAR':'ARG', 'DAS':'ASP', 'DCY':'CYS', 'DGL':'GLU', 'DGN':'GLN', 'DHA':'ALA',
    'DHI':'HIS', 'DIL':'ILE', 'DIV':'VAL', 'DLE':'LEU', 'DLY':'LYS', 'DNP':'ALA', 'DPN':'PHE', 'DPR':'PRO', 'DSN':'SER', 'DSP':'ASP',
    'DTH':'THR', 'DTR':'TRP', 'DTY':'TYR', 'DVA':'VAL', 'EFC':'CYS', 'FLA':'ALA', 'FME':'MET', 'GGL':'GLU', 'GL3':'GLY', 'GLZ':'GLY',
    'GMA':'GLU', 'GSC':'GLY', 'HAC':'ALA', 'HAR':'ARG', 'HIC':'HIS', 'HIP':'HIS', 'HMR':'ARG', 'HPQ':'PHE', 'HTR':'TRP', 'HYP':'PRO',
    'IAS':'ASP', 'IIL':'ILE', 'IYR':'TYR', 'KCX':'LYS', 'LLP':'LYS', 'LLY':'LYS', 'LTR':'TRP', 'LYM':'LYS', 'LYZ':'LYS', 'MAA':'ALA', 'MEN':'ASN',
    'MHS':'HIS', 'MIS':'SER', 'MLE':'LEU', 'MPQ':'GLY', 'MSA':'GLY', 'MSE':'MET', 'MVA':'VAL', 'NEM':'HIS', 'NEP':'HIS', 'NLE':'LEU',
    'NLN':'LEU', 'NLP':'LEU', 'NMC':'GLY', 'OAS':'SER', 'OCS':'CYS', 'OMT':'MET', 'PAQ':'TYR', 'PCA':'GLU', 'PEC':'CYS', 'PHI':'PHE',
    'PHL':'PHE', 'PR3':'CYS', 'PRR':'ALA', 'PTR':'TYR', 'PYX':'CYS', 'SAC':'SER', 'SAR':'GLY', 'SCH':'CYS', 'SCS':'CYS', 'SCY':'CYS',
    'SEL':'SER', 'SEP':'SER', 'SET':'SER', 'SHC':'CYS', 'SHR':'LYS', 'SMC':'CYS', 'SOC':'CYS', 'STY':'TYR', 'SVA':'SER', 'TIH':'ALA',
    'TPL':'TRP', 'TPO':'THR', 'TPQ':'ALA', 'TRG':'LYS', 'TRO':'TRP', 'TYB':'TYR', 'TYI':'TYR', 'TYQ':'TYR', 'TYS':'TYR', 'TYY':'TYR',
    # Map to Self
    'CYS':'CYS',
    'ASP':'ASP',
    'SER':'SER',
    'GLN':'GLN',
    'LYS':'LYS',
    'ILE':'ILE',
    'PRO':'PRO',
    'THR':'THR',
    'PHE':'PHE',
    'ASN':'ASN',
    'GLY':'GLY',
    'HIS':'HIS',
    'LEU':'LEU',
    'ARG':'ARG',
    'TRP':'TRP',
    'ALA':'ALA',
    'VAL':'VAL',
    'GLU':'GLU',
    'TYR':'TYR',
    'MET':'MET'
}

In [4]:
fasta_seqs = {}
for i, r in complexes.iterrows():
    fasta_seqs[f"{r.pdb_id}"] = r.receptor_seq

In [5]:
def get_pdb_seq(pdb_file):
    pdb_structure = pdb_parser.get_structure("peptide", pdb_file)[0]["A"]
    aa_seq = []
    for r in pdb_structure.get_list():
        res_name = r.get_resname()
        if(res_name not in d3to1.keys()):
        # if(res_name not in d3to1.keys()):
            aa_seq.append("X")
        else:
            aa_seq.append(d3to1[
                non_standard_residue_substitutions[res_name]])
    aa_seq = "".join(aa_seq)
    return aa_seq

def get_pdb_xyz(pdb_file):
    """
    get the coordinates
    """
    current_pos = -1000
    X = []
    current_aa = {}  # N, CA, C, O, R
    with open(pdb_file, "r") as f:
        lines = f.readlines()
    for line in lines:
        # print(line)
        if (
            line[0:4].strip() == "ATOM" and int(line[22:26].strip()) != current_pos
        ) or line[0:4].strip() == "TER":
            if current_aa != {}:
                R_group = []
                for atom in current_aa:
                    if atom not in ["N", "CA", "C", "O"]:
                        R_group.append(current_aa[atom])
                if R_group == []:
                    R_group = [current_aa["CA"]]
                R_group = np.array(R_group).mean(0)
                X.append(
                    [
                        current_aa["N"],
                        current_aa["CA"],
                        current_aa["C"],
                        current_aa["O"],
                        R_group,
                    ]
                )
                current_aa = {}
            if line[0:4].strip() != "TER":
                current_pos = int(line[22:26].strip())

        if line[0:4].strip() == "ATOM":
            atom = line[13:16].strip()
            if atom != "H":
                xyz = np.array(
                    [line[30:38].strip(), line[38:46].strip(), line[46:54].strip()]
                ).astype(np.float32)
                current_aa[atom] = xyz
    return np.array(X)

def match_dssp(seq, dssp, ref_seq):
    alignments = pairwise2.align.globalxx(ref_seq, seq)
    ref_seq = alignments[0].seqA
    seq = alignments[0].seqB
    padded_item = np.zeros(9)

    new_dssp = []
    for aa in seq:
        if aa == "-":
            new_dssp.append(padded_item)
        else:
            new_dssp.append(dssp.pop(0))

    matched_dssp = []
    for i in range(len(ref_seq)):
        if ref_seq[i] == "-":
            continue
        matched_dssp.append(new_dssp[i])

    return matched_dssp

def process_dssp(dssp_file):
    aa_type = "ACDEFGHIKLMNPQRSTVWY"
    SS_type = "HBEGITSC"
    rASA_std = [
        115,
        135,
        150,
        190,
        210,
        75,
        195,
        175,
        200,
        170,
        185,
        160,
        145,
        180,
        225,
        115,
        140,
        155,
        255,
        230,
    ]

    with open(dssp_file, "r") as f:
        lines = f.readlines()

    seq = ""
    dssp_feature = []

    p = 0
    while lines[p].strip()[0] != "#":
        p += 1
    for i in range(p + 1, len(lines)):
        aa = lines[i][13]
        if aa == "!" or aa == "*":
            continue
        seq += aa
        SS = lines[i][16]
        if SS == " ":
            SS = "C"
        SS_vec = np.zeros(8)
        SS_vec[SS_type.find(SS)] = 1
        ACC = float(lines[i][34:38].strip())
        ASA = min(1, ACC / rASA_std[aa_type.find(aa)])
        dssp_feature.append(np.concatenate((np.array([ASA]), SS_vec)))

    return seq, dssp_feature

In [ ]:
pdb_files = sorted([
    i for  i in os.listdir("./structure")
    if "receptor.pdb" in i
])

xyz_mismatch = []
dssp_mismatch = []

for f in tqdm(pdb_files):
    pdb_id = f.split("_")[0]
    dssp_path = os.path.join("./dssp", f)
    structure_path = os.path.join("./structure", f)
    try:
        pdb_xyz = get_pdb_xyz(structure_path)
        pdb_seq = get_pdb_seq(structure_path)
        if(len(pdb_seq) != pdb_xyz.shape[0]):
            print("XYZ & Seq Mismatch")
            xyz_mismatch.append(f)
            continue
        os.system(
            f"{DSSP} -i {structure_path} -o {dssp_path}"
        )
        dssp_seq, dssp_matrix = process_dssp(dssp_path)
        if dssp_seq != pdb_seq:
            dssp_matrix = match_dssp(dssp_seq, dssp_matrix, pdb_seq)
        dssp_matrix = np.stack(dssp_matrix)
        filename = f.split('.')[0]
        if(len(pdb_xyz)!= dssp_matrix.shape[0]):
            print("DSSP mismatch")
            dssp_mismatch.append(f)
            continue
        torch.save(
            dssp_matrix,
            os.path.join("./dssp", f"{filename}.pt"),
        )
        torch.save(
            pdb_xyz,
            os.path.join("./structure", f"{filename}.pt"),
        )
    except:
        pass

In [8]:
def match_emb(seq, emb, ref_seq):
    alignments = pairwise2.align.globalxx(ref_seq, seq)
    ref_seq = alignments[0].seqA
    seq = alignments[0].seqB
    padded_item = np.zeros((1024))
    new_emb = []
    count = 0
    for aa in seq:
        if aa == "-":
            new_emb.append(padded_item)
        else:
            new_emb.append(emb[count])
            count += 1
    matched_emb = []
    for i in range(len(ref_seq)):
        if ref_seq[i] == "-":
            continue
        matched_emb.append(new_emb[i])

    return matched_emb

def match_mask(seq, mask, ref_seq):
    alignments = pairwise2.align.globalxx(ref_seq, seq)
    ref_seq = alignments[0].seqA
    seq = alignments[0].seqB
    padded_item = 0.0
    new_mask = []
    count = 0
    for aa in seq:
        if aa == "-":
            new_mask.append(padded_item)
        else:
            new_mask.append(mask[count])
            count += 1
    matched_mask = []
    for i in range(len(ref_seq)):
        if ref_seq[i] == "-":
            continue
        matched_mask.append(new_mask[i])
    return matched_mask

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(PROTT5_PATH, do_lower_case=False,)
emb_model = T5EncoderModel.from_pretrained(PROTT5_PATH)
emb_model = emb_model.to("cuda:0")
emb_model = emb_model.eval()
gc.collect()

def get_emb(seq):
    seq = [" ".join(seq)]
    seq = [re.sub(r"[UZOB]", "X", sequence) for sequence in seq]
    ids = tokenizer.batch_encode_plus(seq, add_special_tokens=True, padding=True)
    input_ids = torch.tensor(ids['input_ids']).to("cuda:0")
    attention_mask = torch.tensor(ids['attention_mask']).to("cuda:0")
    with torch.no_grad():
        embedding = emb_model(input_ids=input_ids,attention_mask=attention_mask)
    embedding = embedding.last_hidden_state.cpu().numpy()
    seq_len = (attention_mask[0] == 1).sum()
    embedding = embedding[0][:seq_len-1]
    return embedding

/home/liangpu/miniforge3/envs/torch2.3/lib/python3.12/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
for f in tqdm(pdb_files):
    structure_path = os.path.join("./structure", f)
    pdb_id = f.split("_")[0]
    receptor_seq = fasta_seqs[pdb_id]
    pdb_seq = get_pdb_seq(structure_path)
    emb = get_emb(receptor_seq)
    emb = np.stack(match_emb(receptor_seq, emb, pdb_seq))
    if(len(pdb_seq) != emb.shape[0]):
        print("Receptor Mismatch")
        continue
    complex_record = complexes.loc[
        complexes.pdb_id==pdb_id
    ].iloc[0]
    ligand_seq = complex_record.ligand_seq
    ligand_emb = get_emb(ligand_seq)
    if(len(ligand_seq) != ligand_emb.shape[0]):
        print("Ligand EMB Mismatch")
        continue
    pocket_mask = np.zeros(len(receptor_seq))
    pocket_index = [
        int(i[1:])-1 
        for i in complex_record.binding_site_reorder.split()
    ]
    pocket_mask[pocket_index] = 1
    pocket_mask = np.stack(match_mask(receptor_seq, pocket_mask, pdb_seq))
    res = {
        "receptor_seq": pdb_seq,
        "receptor_emb": emb,
        "ligand_seq": ligand_seq,
        "ligand_emb": ligand_emb,
        "pocket_mask": pocket_mask
    }
    torch.save(res, f"./seq_emb/{pdb_id}.pt")

In [ ]:
for pdb_id in tqdm(train_ids+val_ids+test_ids): # check feature length
    structure_feat = torch.load(os.path.join(
        "./structure", f"{pdb_id}_receptor.pt"
    ))
    dssp_feat = torch.load(os.path.join(
        "./dssp", f"{pdb_id}_receptor.pt"
    ))
    seq_feat = torch.load(os.path.join(
        "./seq_emb", f"{pdb_id}.pt"
    ))
    assert structure_feat.shape[0] == dssp_feat.shape[0] == seq_feat["receptor_emb"].shape[0]

100%|██████████| 5605/5605 [00:50<00:00, 111.29it/s]
